# Bengali LLM — Colab Inference Server
## Built with Unsloth · FastAPI · ngrok
---
**Run cells 1 → 7 in order.**

You only need to update two things before running:

**Cell 3** → set `ADAPTER_PATH` to your LoRA adapter folder on Google Drive

**Cell 7** → set `NGROK_AUTH_TOKEN` and `NGROK_STATIC_DOMAIN` from your ngrok dashboard

In [ ]:
# ── Cell 1: Install dependencies ────────────────────────────
# Unsloth bundles compatible versions of transformers/peft/bitsandbytes.
# Install it first so there are no version conflicts.

!pip install -q unsloth
!pip install -q fastapi==0.115.4 uvicorn==0.32.0 pyngrok==7.2.2 nest_asyncio==1.6.0

# Mount Google Drive to access your saved LoRA adapter
from google.colab import drive
drive.mount('/content/drive')

print('\u2705 Dependencies installed and Drive mounted.')

In [ ]:
# ── Cell 2: Imports ─────────────────────────────────────────
import torch
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Optional
from unsloth import FastLanguageModel
from peft import PeftModel

nest_asyncio.apply()  # lets uvicorn run inside Colab's event loop
print('\u2705 Imports done.')

In [ ]:
# ── Cell 3: Load Model ──────────────────────────────────────

# Base model — the same one used during fine-tuning
BASE_MODEL_ID = 'unsloth/Qwen2.5-3B-Instruct'

# ↓ UPDATE THIS: path to your LoRA adapter folder on Google Drive
ADAPTER_PATH  = '/content/drive/MyDrive/your_adapter_folder'

# Must match max_seq_length used during fine-tuning
MAX_SEQ_LENGTH = 2048

print('Loading base model with Unsloth (4-bit QLoRA)...')

# Step 1: Load base model via Unsloth — applies the same internal patches
# as during training, so the LoRA adapter weights will match exactly.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,    # auto: float16 on T4, bfloat16 on A100+
    load_in_4bit   = True,    # 4-bit quantisation — same as training
)

# Step 2: Attach your fine-tuned LoRA adapter
# Unsloth saves in standard PEFT format, so PeftModel.from_pretrained works.
print('Loading LoRA adapter from Drive...')
model = PeftModel.from_pretrained(model, ADAPTER_PATH)

# Step 3: Enable Unsloth's optimised inference kernel (~2x faster generation)
FastLanguageModel.for_inference(model)

print('\u2705 Model ready! Device:', next(model.parameters()).device)

In [ ]:
# ── Cell 4: System Prompt ───────────────────────────────────
# Prepended as the system message before every conversation.
# Edit to match the prompt style used during fine-tuning.

SYSTEM_PROMPT = (
    'You are a helpful Bengali language assistant developed by the '
    'Calcutta University Data Science Lab. You are fine-tuned on Bengali '
    'datasets covering reading comprehension, government schemes, history, '
    'technology, and everyday Bengali conversation. '
    'Always respond in the same language as the user\'s question. '
    'Be concise, factual, and helpful.'
)
print('\u2705 System prompt set.')

In [ ]:
# ── Cell 5: Inference Function ──────────────────────────────

def run_inference(
    question: str,
    conversation_history: List[Dict[str, str]],
    max_new_tokens: int = 512,
    temperature: float = 0.7,
    top_p: float = 0.9,
) -> str:
    """
    Builds the Qwen2.5 chat template prompt, runs inference via Unsloth,
    and returns ONLY the newly generated assistant reply.

    Args:
        question             : current user message
        conversation_history : list of {role, content} for prior turns
                               (sent automatically by the LangGraph backend)
    """
    # 1. Build messages: system -> prior history -> new question
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    messages.extend(conversation_history)
    messages.append({'role': 'user', 'content': question})

    # 2. Apply Qwen2.5 chat template
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # 3. Tokenise and move to model device
    inputs = tokenizer([prompt_text], return_tensors='pt').to(model.device)
    input_length = inputs['input_ids'].shape[1]

    # 4. Generate — Unsloth's for_inference() speeds this up significantly
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = temperature,
            top_p              = top_p,
            do_sample          = True,
            repetition_penalty = 1.1,  # prevents repetition loops
            pad_token_id       = tokenizer.eos_token_id,
        )

    # 5. Decode only NEW tokens — strip the input prompt from output
    new_tokens = output_ids[0][input_length:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response


# --- Optional quick sanity test (uncomment to run before starting server) ---
# test = run_inference('বাংলাদেশের রাজধানী কী?', [])
# print('Test reply:', test)

print('\u2705 Inference function defined.')

In [ ]:
# ── Cell 6: FastAPI App ─────────────────────────────────────

app = FastAPI(title='Bengali LLM Colab Inference')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)


class InferenceRequest(BaseModel):
    question: str
    conversation_history: Optional[List[Dict[str, str]]] = []
    max_new_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.7


class InferenceResponse(BaseModel):
    response: str


@app.get('/health')
def health():
    """Liveness check — polled by the LangGraph backend on startup."""
    return {'status': 'ok', 'model': 'Qwen2.5-3B Bengali Fine-tuned (Unsloth)'}


@app.post('/generate', response_model=InferenceResponse)
def generate(req: InferenceRequest):
    """
    Main inference endpoint called by the LangGraph backend.
    Receives: { question, conversation_history, max_new_tokens, temperature }
    Returns:  { response: '...' }
    """
    try:
        result = run_inference(
            question             = req.question,
            conversation_history = req.conversation_history or [],
            max_new_tokens       = req.max_new_tokens or 512,
            temperature          = req.temperature or 0.7,
        )
        return InferenceResponse(response=result)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print('\u2705 FastAPI app defined.  Endpoints: GET /health   POST /generate')

In [ ]:
# ── Cell 7: Start ngrok + uvicorn ───────────────────────────
from pyngrok import ngrok, conf

# ↓ UPDATE THESE TWO VALUES
NGROK_AUTH_TOKEN    = 'your_ngrok_auth_token_here'          # https://dashboard.ngrok.com/authtokens
NGROK_STATIC_DOMAIN = 'your-static-domain.ngrok-free.app'  # https://dashboard.ngrok.com/domains

conf.get_default().auth_token = NGROK_AUTH_TOKEN

listener = ngrok.connect(
    addr   = 8001,
    domain = NGROK_STATIC_DOMAIN,  # static domain -> URL never changes between Colab restarts
)

print(f'\u2705 Public URL  : {listener.public_url}')
print(f'   /generate  : {listener.public_url}/generate')
print(f'   /health    : {listener.public_url}/health')
print()
print('>>> Copy the /generate URL into backend/.env as COLAB_ENDPOINT')

# Run uvicorn in a daemon thread so this cell does not block
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8001, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print('\U0001f680 Inference server is RUNNING!')